In [ ]:
!pip install kaggle

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"naghamzidiah","key":"85ec7f976b55ece3708564fd6417ad97"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kwentar/blur-dataset

Dataset URL: https://www.kaggle.com/datasets/kwentar/blur-dataset
License(s): CC0-1.0
100% 1.49G/1.49G [00:15<00:00, 102MB/s]



In [ ]:
!unzip blur-dataset.zip

Archive:  blur-dataset.zip
  inflating: blur_dataset_scaled/defocused_blurred/0_IPHONE-SE_F.JPG  
  inflating: blur_dataset_scaled/defocused_blurred/100_NIKON-D3400-35MM_F.JPG  
  inflating: blur_dataset_scaled/defocused_blurred/101_NIKON-D3400-35MM_F.JPG  
  inflating: blur_dataset_scaled/defocused_blurred/102_NIKON-D3400-35MM_F.JPG  
  inflating: blur_dataset_scaled/defocused_blurred/103_HUAWEI-P20_F.jpg  
  inflating: blur_dataset_scaled/defocused_blurred/104_IPHONE-SE_F.jpg  
  inflating: blur_dataset_scaled/defocused_blurred/105_IPHONE-SE_F.jpg  
  inflating: blur_dataset_scaled/defocused_blurred/106_NIKON-D3400-35MM_F.JPG  
  inflating: blur_dataset_scaled/defocused_blurred/107_XIAOMI-MI8-SE_F.jpg  
  inflating: blur_dataset_scaled/defocused_blurred/108_XIAOMI-MI8-SE_F.jpg  
  inflating: blur_dataset_scaled/defocused_blurred/109_HONOR-7X_F.jpg  
  inflating: blur_dataset_scaled/defocused_blurred/10_ASUS-ZENFONE-LIVE-ZB501KL_F.jpg  
  inflating: blur_dataset_scaled/defocused_blurr

In [ ]:
!ls

blur_dataset_scaled  defocused_blurred	kaggle.json	sample_data
blur-dataset.zip     drive		motion_blurred	sharp


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p /content/drive/MyDrive/BlurProject

In [ ]:
!cp -r /content/sharp /content/drive/MyDrive/BlurProject/
!cp -r /content/motion_blurred /content/drive/MyDrive/BlurProject/
!cp -r /content/defocused_blurred /content/drive/MyDrive/BlurProject/

## Dataset Preparation

In this step, we prepare the dataset for training.

We will:
- Load images from the dataset folders.
- Convert images to grayscale.
- Resize them to a fixed size (256×256).
- Assign labels:
  - Sharp images → 0
  - Blurred images (defocused + motion) → 1

This step ensures that all images are consistent and ready for feature extraction.

In [ ]:
import os
import cv2
import numpy as np
import random

import random
import numpy as np

random.seed(42)
np.random.seed(42)

# Number of images to use (to keep training fast)
num_sharp = 300
num_blur = 300   # 300 defocused + 300 motion

sharp_path = "/content/drive/MyDrive/BlurProject/sharp"
defocus_path = "/content/drive/MyDrive/BlurProject/defocused_blurred"
motion_path = "/content/drive/MyDrive/BlurProject/motion_blurred"

images = []
labels = []

def load_images(folder, label, max_images):
    files = os.listdir(folder)
    random.shuffle(files)

    for file in files[:max_images]:
        img_path = os.path.join(folder, file)
        img = cv2.imread(img_path)

        if img is not None:
            # Convert to grayscale
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Resize to fixed size
            img = cv2.resize(img, (256, 256))

            images.append(img)
            labels.append(label)

# Load images
load_images(sharp_path, 0, num_sharp)
load_images(defocus_path, 1, num_blur)
load_images(motion_path, 1, num_blur)

print("Total images loaded:", len(images))
print("Sharp images:", labels.count(0))
print("Blur images:", labels.count(1))

Total images loaded: 900
Sharp images: 300
Blur images: 600


## Feature Extraction

In this step, we extract handcrafted features from each image
to capture blur characteristics.

We will compute features from:

- Spatial domain (edges, gradients, contrast)
- Frequency domain (high-frequency energy)

Each image will be converted into a numerical feature vector
that will be used as input to the MLP classifier.

## 1. Variance of Laplacian Feature

The Variance of Laplacian is a sharpness measure based on edge intensity.

The Laplacian operator detects edges by computing the second derivative
of the image intensity.

Sharp images contain strong edges, which result in higher variance values.
Blurred images have weaker edges, leading to lower variance.

This feature is widely used for blur detection due to its simplicity
and effectiveness.

In [ ]:
def variance_of_laplacian(image):
    return cv2.Laplacian(image, cv2.CV_64F).var()

# Test on first image
print("Example Laplacian variance:", variance_of_laplacian(images[0])) #  represents the variance of edge intensity in the image. Higher values indicate stronger edges and therefore sharper images.

Example Laplacian variance: 1639.5245909541845


## 2. Tenengrad Feature (Sobel Gradient)

The Tenengrad method measures image sharpness based on gradient magnitude.
It uses the Sobel operator to detect edge strength in horizontal and vertical directions.

Sharper images have stronger gradients,
while blurred images produce weaker gradient responses.

In [ ]:
def tenengrad(image):
    # Sobel gradients
    sobel_x = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)

    # Gradient magnitude
    gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)

    # Return mean gradient magnitude
    return np.mean(gradient_magnitude)

print("Tenengrad example:", tenengrad(images[0]))

Tenengrad example: 56.0762651307448


## 3. Image Contrast Feature

Image contrast measures the intensity variation within an image.

Blurred images tend to smooth intensity transitions,
which reduces contrast variation.

We compute contrast using the standard deviation of pixel intensities.
Higher standard deviation indicates higher contrast.

In [ ]:
def image_contrast(image):
    # Standard deviation of pixel intensities
    return np.std(image)

print("Contrast example:", image_contrast(images[0]))

Contrast example: 74.79194667455084


## 4. High-Frequency Energy (FFT Feature)

Blur affects the frequency components of an image.

Sharp images contain strong high-frequency components
(due to edges and fine details).

Blurred images suppress high-frequency components
and emphasize low-frequency components.

We compute the high-frequency energy using the Fast Fourier Transform (FFT).

In [ ]:
def high_frequency_energy(image):
    # Apply FFT
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)

    # Magnitude spectrum
    magnitude_spectrum = np.abs(fshift)

    # Create high-frequency mask (remove center low frequencies)
    rows, cols = image.shape
    crow, ccol = rows // 2, cols // 2

    mask = np.ones((rows, cols))
    r = 30  # radius for low-frequency removal
    mask[crow-r:crow+r, ccol-r:ccol+r] = 0

    # Apply mask
    high_freq = magnitude_spectrum * mask

    # Return average high-frequency energy
    return np.mean(high_freq)

print("FFT High-Frequency Energy example:", high_frequency_energy(images[0]))

FFT High-Frequency Energy example: 2697.7184639629118


## Feature Vector Construction

In this step, we combine all extracted features into a single feature vector
for each image.

Each image will be represented by:

1. Variance of Laplacian
2. Tenengrad (Gradient Magnitude)
3. Image Contrast
4. High-Frequency Energy (FFT)

This feature vector will serve as the input to the MLP classifier.

In [ ]:
# Create feature matrix
feature_list = []

for img in images:
    lap = variance_of_laplacian(img)
    ten = tenengrad(img)
    con = image_contrast(img)
    fft = high_frequency_energy(img)

    feature_list.append([lap, ten, con, fft])

X = np.array(feature_list)
y = np.array(labels)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)

Feature matrix shape: (900, 4)
Labels shape: (900,)


## Data Splitting and Normalization

Before training the neural network, we perform:

1. Train-Test split to evaluate model performance.
2. Feature normalization to ensure all features are on a similar scale.

Normalization is important because neural networks are sensitive
to feature scale differences.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split dataset (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=42, stratify=y )

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 630
Testing samples: 270


## MLP Model Architecture

We implement a traditional Multilayer Perceptron (MLP) classifier.

Architecture:
- Input layer: 4 handcrafted features
- Hidden Layer 1: 32 neurons with ReLU activation
- Hidden Layer 2: 16 neurons with ReLU activation
- Output Layer: 1 neuron with sigmoid activation

Loss Function: Binary Cross-Entropy  
Optimizer: Adam

The model learns to classify images as Sharp (0) or Blur (1).

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Build MLP model
model_adam = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)

# Train model
model_adam.fit(X_train, y_train)

# Predictions
y_pred = model_adam.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9481481481481482

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.97      0.93        90
           1       0.98      0.94      0.96       180

    accuracy                           0.95       270
   macro avg       0.94      0.95      0.94       270
weighted avg       0.95      0.95      0.95       270


Confusion Matrix:
 [[ 87   3]
 [ 11 169]]


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


## Optimizer Comparison (Adam vs SGD)

To evaluate the impact of different optimization algorithms,
we compare Adam and Stochastic Gradient Descent (SGD)
using the same network architecture.

This experiment helps analyze convergence behavior
and classification performance differences.

In [ ]:
# Build MLP model with SGD
model_sgd = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='sgd',
    max_iter=300,
    random_state=42
)

# Train model
model_sgd.fit(X_train, y_train)

# Predictions
y_pred = model_sgd.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9259259259259259

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.91      0.89        90
           1       0.95      0.93      0.94       180

    accuracy                           0.93       270
   macro avg       0.91      0.92      0.92       270
weighted avg       0.93      0.93      0.93       270


Confusion Matrix:
 [[ 82   8]
 [ 12 168]]


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Adam achieved higher accuracy and better convergence compared to SGD. This is because Adam adapts the learning rate using first and second moment estimates, which improves stability and convergence speed.

## Live Demo: Testing New Images

To demonstrate the model performance during the presentation,
we implement a function that:

1. Loads a new image.
2. Applies the same preprocessing steps.
3. Extracts handcrafted features.
4. Normalizes the features using the trained scaler.
5. Uses the trained MLP model to predict whether the image is Sharp or Blur.

This ensures consistency between training and testing pipelines.

In [ ]:
def predict_image(image_path):
    # Load image
    img = cv2.imread(image_path)

    if img is None:
        print("Error: Could not load image.")
        return

    # Convert to grayscale
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_gray = cv2.resize(img_gray, (256, 256))

    # Extract features
    lap = variance_of_laplacian(img_gray)
    ten = tenengrad(img_gray)
    con = image_contrast(img_gray)
    fft = high_frequency_energy(img_gray)

    features = np.array([[lap, ten, con, fft]])

    # Normalize using trained scaler
    features_scaled = scaler.transform(features)

    # Predict
    # Using Adam model for final prediction (best performing model)
    prediction = model_adam.predict(features_scaled)[0]
    probability = model_adam.predict_proba(features_scaled)[0][prediction]

    # Output result
    if prediction == 0:
        print("Prediction: Sharp")
    else:
        print("Prediction: Blur")

    print("Confidence:", round(probability, 4))

In [ ]:
# test case 1
predict_image("/content/drive/MyDrive/BlurProject/sharp/110_IPHONE-7_S.jpeg")

Prediction: Sharp
Confidence: 0.9933


In [ ]:
# test case 2
predict_image("/content/drive/MyDrive/BlurProject/motion_blurred/105_IPHONE-SE_M.jpg")

Prediction: Blur
Confidence: 0.9778
